In [ ]:
#Step 1:Creation of files with coordinates and velocity components
import os
import glob
import meshio  # Ensure meshio is installed (pip install meshio)

# Folder paths
vtu_folder = r"Your Path"
output_folder = r"Your Path\Step_1_CFD_to_MRI\Support_files\Location_&_velocity_data"

# Ensure the output directory exists
os.makedirs(output_folder, exist_ok=True)

# Generate a list of all VTU files,(start_point,end_point,increment)
vtu_files = [os.path.join(vtu_folder, f'all_results_{str(i).zfill(5)}.vtu') for i in range(0, 99, 1)]

# Process each VTU file
for vtu_file in vtu_files:
    # Load the VTU file
    mesh = meshio.read(vtu_file)
    
    # Extract velocity data and point coordinates
    velocity = mesh.point_data['velocity']
    points = mesh.points
    
    # Extract timestep number from the file name
    timestep = os.path.basename(vtu_file).split('_')[-1].split('.')[0]
    
    # Create a text file to store coordinates and velocities
    output_file_path = os.path.join(output_folder, f'coor_vel_{timestep}.txt')
    
    # Write coordinates and velocity data to the text file
    with open(output_file_path, 'w') as f:
        f.write("X,Y,Z,U,V,W\n")  # Write header
        for point, vel in zip(points, velocity):
            f.write(f"{point[0]},{point[1]},{point[2]},{vel[0]},{vel[1]},{vel[2]}\n")
    
    print(f"Data saved to: {output_file_path}")

In [ ]:
#Step 2:h5_file_creation
import numpy as np
import h5py
import os

# Define the bounds and the number of slices for each dimension (taken from the 4D flow MRI images)
x_bounds = (-7.96, 17.52)
y_bounds = (-19.47, 14.15)
z_bounds = (-10.39, 19.46)

num_x_slices = 72
num_y_slices = 192
num_z_slices = 192
num_timesteps = 99  # From start_point to end_point, inclusive, in increments of (increment each time)

# Generate grid points for each dimension
x_slices = np.linspace(x_bounds[0], x_bounds[1], num_x_slices)
y_slices = np.linspace(y_bounds[0], y_bounds[1], num_y_slices)  
z_slices = np.linspace(z_bounds[0], z_bounds[1], num_z_slices)

# Initialize 4D arrays for u, v, and w velocities (time, x, y, z)
u_velocity_array = np.zeros((num_timesteps, num_x_slices, num_y_slices, num_z_slices))
v_velocity_array = np.zeros((num_timesteps, num_x_slices, num_y_slices, num_z_slices))
w_velocity_array = np.zeros((num_timesteps, num_x_slices, num_y_slices, num_z_slices))

# Initialize arrays for maximum velocity values for each timestep
u_max_values = np.zeros(num_timesteps, dtype=np.float32)
v_max_values = np.zeros(num_timesteps, dtype=np.float32)
w_max_values = np.zeros(num_timesteps, dtype=np.float32)

# Folder where the text files are located
folder_path = r"Your Path\Step_1_CFD_to_MRI\Support_files\Location_&_velocity_data"

# Loop through each timestep and load velocity data from the text files
for t, step in enumerate(range(0, 99, 1)):  # Loop for filenames from start_time to end_time in increments of (whatever you like)
    # Construct the file path for the current timestep
    file_path = os.path.join(folder_path, f'coor_vel_{step:05d}.txt')
    
    # Load velocity data from the text file (make sure each file has the format as the original)
    data = np.loadtxt(file_path, delimiter=',', skiprows=1)

    # Extract coordinates and velocity components
    x_coords = data[:, 0]
    y_coords = data[:, 1]
    z_coords = data[:, 2]
    u_velocity = data[:, 3]
    v_velocity = data[:, 4]
    w_velocity = data[:, 5]

    # Populate the 4D arrays with the corresponding velocity values for the current timestep
    for i in range(len(data)):
        # Find the indices for the current coordinate values
        x_index = np.argmin(np.abs(x_slices - x_coords[i]))
        y_index = np.argmin(np.abs(y_slices - y_coords[i]))
        z_index = np.argmin(np.abs(z_slices - z_coords[i]))

        # Assign velocity values to the appropriate position in the arrays (x, y, z) for timestep t
        u_velocity_array[t, x_index, y_index, z_index] = u_velocity[i]
        v_velocity_array[t, x_index, y_index, z_index] = v_velocity[i]
        w_velocity_array[t, x_index, y_index, z_index] = w_velocity[i]
    
    # Calculate the max values for this timestep and store them
    u_max_values[t] = np.max(u_velocity_array[t])  # Maximum for u at timestep t
    v_max_values[t] = np.max(v_velocity_array[t])  # Maximum for v at timestep t
    w_max_values[t] = np.max(w_velocity_array[t])  # Maximum for w at timestep t

# Create the HDF5 file 
h5_file_path = rh5_file_path = r"Your Path"

with h5py.File(h5_file_path, 'w') as h5_file:
    # Create dx dataset
    dx = np.array([x_slices[1] - x_slices[0], y_slices[1] - y_slices[0], z_slices[1] - z_slices[0]], dtype=np.float32)
    h5_file.create_dataset('dx', data=dx.reshape(1, 3), dtype='float32')

    # Create mask with all values set to 1 and the same shape as u, v, w arrays in the xyz
    mask_shape = (1, num_x_slices, num_y_slices, num_z_slices)
    mask = np.full(mask_shape, 1, dtype=np.float32)  # All values set to 1
    h5_file.create_dataset('mask', data=mask)

    # Create u, v, and w datasets with the correct shapes
    h5_file.create_dataset('u', data=u_velocity_array, dtype='float32')
    h5_file.create_dataset('v', data=v_velocity_array, dtype='float32')
    h5_file.create_dataset('w', data=w_velocity_array, dtype='float32')

    # Store maximum values for each velocity component over all timesteps
    h5_file.create_dataset('u_max', data=u_max_values, dtype='float32')  # Changed to save array of max values
    h5_file.create_dataset('v_max', data=v_max_values, dtype='float32')  # Changed to save array of max values
    h5_file.create_dataset('w_max', data=w_max_values, dtype='float32')  # Changed to save array of max values

# Output the shapes of the resulting arrays
print("Shape of u_velocity_array:", u_velocity_array.shape)
print("Shape of v_velocity_array:", v_velocity_array.shape)
print("Shape of w_velocity_array:", w_velocity_array.shape)
print(f"HDF5 file created at: {h5_file_path}")

# Print the maximum values for each velocity component
print(f"Maximum u values: {u_max_values}")
print(f"Maximum v values: {v_max_values}")
print(f"Maximum w values: {w_max_values}")

print(f"HDF5 file successfully created at: {h5_file_path}")

In [ ]:
#Step 3:h5_file_inspection and initial mask values extraction
import h5py
import numpy as np

def print_hdf5_contents_and_check_mask(file_path, output_txt_path):
    # Function to recursively print the contents of an h5 file
    def print_attrs(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name}")
            print(f"Shape: {obj.shape}")
            print(f"Data: {obj[...][:10]}\n")  # Prints the first 10 elements of the dataset for brevity

            # Check if the dataset is the mask array
            if name == "mask":  # Assuming the mask array is named "mask"
                print(f"Checking if all values in 'mask' are zero...")
                mask_values = obj[...]

                # Print the number of values in the mask array
                print(f"Number of values in the mask array: {mask_values.size}")

                # Write the mask values to a text file, flattening it first
                with open(output_txt_path, 'w') as f:
                    np.savetxt(f, mask_values.flatten(), fmt='%d')  # Flatten mask to 1D before saving

                if np.all(mask_values == 0):
                    print("All values in the mask array are zero.")
                else:
                    print("The mask array contains non-zero values.")
                    print(f"Mask values written to: {output_txt_path}")
        elif isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        else:
            print(f"Unknown type: {name}")

    # Open the HDF5 file
    with h5py.File(file_path, 'r') as hdf:
        print(f"Opening file: {file_path}")
        # Use the visitor pattern to explore all groups and datasets
        hdf.visititems(print_attrs)

# Provide the path to your HDF5 file and the output text file
file_path = r"Your Path"
output_txt_path = r"Your Path\Step_1_CFD_to_MRI\Support_files\Initial_values_mask.txt"
print_hdf5_contents_and_check_mask(file_path, output_txt_path)

In [ ]:
#Step 4:Velocity_magnitude_array_all_timestpes_creation
import h5py
import numpy as np

def calculate_velocity_magnitude_for_timestep(file_path, timestep_index):
    # Open the HDF5 file and calculate the velocity magnitude for the specific timestep
    with h5py.File(file_path, 'r') as hdf:
        # Load the 'u', 'v', and 'w' arrays for the specific timestep
        u = hdf['u'][timestep_index, ...]  # Load the u array for the specific timestep
        v = hdf['v'][timestep_index, ...]  # Load the v array for the specific timestep
        w = hdf['w'][timestep_index, ...]  # Load the w array for the specific timestep
        
        # Calculate the velocity magnitude
        velocity_magnitude = np.sqrt(u**2 + v**2 + w**2)
        
        # Return the velocity magnitude without reshaping 
        return velocity_magnitude

# Provide the path to your original HDF5 file
original_file_path = r"Your Path"
new_file_path = r"Your Path"

# Number of timesteps
num_timesteps = 99

# Initialize an array to hold all velocity magnitudes for each timestep
all_velocity_magnitudes = np.zeros((num_timesteps, 72, 192, 192))

# Loop through each timestep to calculate and store the velocity magnitude
for timestep_index in range(num_timesteps):
    # Step 1: Calculate the velocity magnitude for the specific timestep
    velocity_magnitude = calculate_velocity_magnitude_for_timestep(original_file_path, timestep_index)
    
    # Step 2: Store the velocity magnitude in the array
    all_velocity_magnitudes[timestep_index] = velocity_magnitude

# Step 3: Save all velocity magnitudes to a new HDF5 file
with h5py.File(new_file_path, 'w') as hdf:
    hdf.create_dataset('velocity_magnitude', data=all_velocity_magnitudes)
    print(f"All velocity magnitudes saved to: {new_file_path}")

# Optional: Print the shape of the velocity magnitude array to verify
print(f"All velocity magnitude array shape: {all_velocity_magnitudes.shape}")

In [ ]:
#Step 5:New_mask_creation
import h5py
import numpy as np

def create_new_mask_array(velocity_magnitude_file_path, output_file_path):
    # Open the HDF5 file containing the velocity magnitude array
    with h5py.File(velocity_magnitude_file_path, 'r') as hdf:
        # Load the velocity magnitude array
        velocity_magnitude = hdf['velocity_magnitude'][...]  
        
    # Initialize an array to hold the new masks for each timestep
    num_timesteps = velocity_magnitude.shape[0]
    new_masks = np.zeros((num_timesteps, 72, 192, 192), dtype=np.uint8)  # Using uint8 for mask values

    # Create a new mask for each timestep based on the velocity magnitude
    for timestep_index in range(num_timesteps):
        # Create a mask: set to 0 where velocity magnitude is 0, and 1 where it's non-zero
        new_masks[timestep_index] = np.where(velocity_magnitude[timestep_index] == 0, 0, 1)

    # Save the new mask array to a new HDF5 file
    with h5py.File(output_file_path, 'w') as hdf_out:
        # Create a dataset for the new masks
        hdf_out.create_dataset('mask', data=new_masks)
        print(f"New mask array saved to: {output_file_path}")

# Provide the path to your velocity magnitude HDF5 file and the new output HDF5 file
velocity_magnitude_file_path = r"Your Path"
output_file_path = r"Your Path"

# Create the new mask array
create_new_mask_array(velocity_magnitude_file_path, output_file_path)

In [ ]:
#Step 6:Time_averaged_mask_creation
import h5py
import numpy as np

def create_time_averaged_mask(input_file_path, output_file_path):
    # Open the HDF5 file to read the mask array
    with h5py.File(input_file_path, 'r') as hdf:
        mask = hdf['mask'][...]  # Load the entire mask array

    # Calculate the time-averaged mask
    time_averaged_mask = np.mean(mask, axis=0)  # Average over the first dimension (timesteps)

    # Reshape to (1, 50, 36, 57)
    time_averaged_mask = time_averaged_mask[np.newaxis, ...]  # Add a new axis for the first dimension

    # Save the new time-averaged mask array to a new HDF5 file
    with h5py.File(output_file_path, 'w') as hdf_out:
        hdf_out.create_dataset('time_averaged_mask', data=time_averaged_mask)
        print(f"Time-averaged mask array saved to: {output_file_path}")

# Example usage
input_file_path = r"Your Path\new_mask.h5"
output_file_path = r"Your Path\new_mask_time_avg.h5"

create_time_averaged_mask(input_file_path, output_file_path)

In [ ]:
#Step 7:Old_with_new_mask_replacement_in_original_h5_file
import h5py
import numpy as np

def replace_mask_from_another_h5(file_path, updated_mask_path):
    # Step 1: Load the updated mask from new_mask.h5
    with h5py.File(updated_mask_path, 'r') as updated_hdf:
        updated_mask = updated_hdf['time_averaged_mask'][...]  # Load the mask array

    # Step 2: Open the original HDF5 file and replace the mask
    with h5py.File(file_path, 'a') as hdf:
        if 'mask' in hdf:
            del hdf['mask']  # Delete the old mask array
            print(f"Old mask deleted from: {file_path}")
        
        # Create a new dataset for the mask array
        hdf.create_dataset('mask', data=updated_mask)
        print(f"New mask array from {updated_mask_path} added to: {file_path}")

def print_hdf5_contents(file_path):
    # Function to recursively print the contents of an HDF5 file
    def print_attrs(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name}")
            print(f"Shape: {obj.shape}")
            print(f"Data (first 10 elements): {obj[...].flatten()[:10]}\n")  # Print first 10 elements for brevity
        elif isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        else:
            print(f"Unknown type: {name}")

    # Step 3: Print the contents of the updated HDF5 file
    with h5py.File(file_path, 'r') as hdf:
        print(f"Opening file: {file_path}")
        hdf.visititems(print_attrs)  # Recursively print all groups and datasets

# File paths for your case
file_path = r"Your Path\velocity_data_simv_v3.h5"
updated_mask_path = r"Your Path\new_mask_time_avg.h5"

# Replace the old mask with the new one from new_mask.h5
replace_mask_from_another_h5(file_path, updated_mask_path)

# Print the contents of the updated HDF5 file
print_hdf5_contents(file_path)

In [ ]:
#Step 8:cm/s_to_m/s_conversion_for_velocity_components
import h5py

def divide_by_100(file_path):
    # Open the HDF5 file in read/write mode
    with h5py.File(file_path, 'a') as hdf:
        # Divide the 'u' dataset by 100
        if 'u' in hdf:
            u_data = hdf['u'][...]  # Load the u array
            u_data /= 100  # Divide by 100
            hdf['u'][...] = u_data  # Update the dataset
            
        # Divide the 'u_max' dataset by 100
        if 'u_max' in hdf:
            u_max_data = hdf['u_max'][...]  # Load the u_max value
            u_max_data /= 100  # Divide by 100
            hdf['u_max'][...] = u_max_data  # Update the dataset

        # Divide the 'v' dataset by 100
        if 'v' in hdf:
            v_data = hdf['v'][...]  # Load the v array
            v_data /= 100  # Divide by 100
            hdf['v'][...] = v_data  # Update the dataset

        # Divide the 'v_max' dataset by 100
        if 'v_max' in hdf:
            v_max_data = hdf['v_max'][...]  # Load the v_max value
            v_max_data /= 100  # Divide by 100
            hdf['v_max'][...] = v_max_data  # Update the dataset

        # Divide the 'w' dataset by 100
        if 'w' in hdf:
            w_data = hdf['w'][...]  # Load the w array
            w_data /= 100  # Divide by 100
            hdf['w'][...] = w_data  # Update the dataset

        # Divide the 'w_max' dataset by 100
        if 'w_max' in hdf:
            w_max_data = hdf['w_max'][...]  # Load the w_max value
            w_max_data /= 100  # Divide by 100
            hdf['w_max'][...] = w_max_data  # Update the dataset

    print(f"Updated the 'u', 'u_max', 'v', 'v_max', 'w', and 'w_max' datasets by dividing all values by 100.")

# Provide the path to your HDF5 file
file_path = r"Your Path\velocity_data_simv_v3.h5"

# Perform the division
divide_by_100(file_path)

# Print the contents of the updated HDF5 file to verify the changes
def print_hdf5_contents(file_path):
    # Function to recursively print the contents of an h5 file
    def print_attrs(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name}")
            print(f"Shape: {obj.shape}")
            print(f"Data (first 10 elements): {obj[...].flatten()[:10]}\n")  # Print first 10 elements for brevity
        elif isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        else:
            print(f"Unknown type: {name}")

    # Open the HDF5 file and print its contents
    with h5py.File(file_path, 'r') as hdf:
        print(f"Opening file: {file_path}")
        hdf.visititems(print_attrs)  # Recursively print all groups and datasets

# Print the contents of the updated HDF5 file
print_hdf5_contents(file_path)